# 02 — Prototype: English ↔ Hiligaynon Chatbot

Test the `HiligaynonChatbot` module interactively.

The chatbot:
1. **Analyzes** the input (language, intent, key text, vowel variants)
2. **Retrieves** relevant dictionary entries (vowel-aware)
3. **Generates** a natural conversational response

In [1]:
import sys, os, importlib
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Load environment
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Configure DSPy with Groq
import dspy
lm = dspy.LM("groq/llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))
dspy.configure(lm=lm)

print(f"Root: {ROOT}")
print(f"LM configured: {lm.model}")

Root: c:\Users\gioan\Documents\GitHub\chatbutt
LM configured: groq/llama-3.3-70b-versatile


## 1. Initialize the Chatbot

In [2]:
from src.modules import HiligaynonChatbot

chatbot = HiligaynonChatbot(chroma_dir=ROOT / "chroma_db")
print(f"Chatbot ready. Collection: {chatbot.collection.name} ({chatbot.collection.count()} entries)")

Chatbot ready. Collection: hiligaynon_dictionary (23656 entries)


## 2. Test: English → Hiligaynon Translation

In [3]:
# English → Hiligaynon
result = chatbot(user_message="How do you say 'good morning' in Hiligaynon?")

print(f"Language: {result.input_language}")
print(f"Intent:   {result.intent}")
print(f"Key text: {result.key_text}")
print(f"\n📝 Analysis:\n{result.analysis}")
print(f"\n💬 Response:\n{result.response}")
print(f"\n📚 Context used ({len(result.context_used)} entries):")
for c in result.context_used[:3]:
    print(f"   {c[:100]}")

Language: english
Intent:   translate
Key text: good morning

📝 Analysis:
The input is in English, and the user is requesting a translation of a specific phrase into Hiligaynon. There are no noticeable vowel variants or unique word structures in the input, as it is a straightforward translation request.

💬 Response:
In Hiligaynon, "good morning" can be translated to "Maayong aga." This is a common and appropriate greeting used during the morning hours. The word "maayong" means good, and "aga" refers to morning. So, when you wake up or meet someone in the morning, you can greet them with "Maayong aga." It's a lovely way to start the day, showing respect and kindness. The Hiligaynon language is rich in expressions of greetings and courtesy, reflecting the culture's strong emphasis on social bonds and community. When using this greeting, remember that Hiligaynon, like many other languages, has its unique vowel variations, but "Maayong aga" remains a standard and widely recognized greeting

## 3. Test: Hiligaynon → English Translation

In [4]:
# Hiligaynon → English
result = chatbot(user_message="Ano ang buot silingon sang 'maayo'?")

print(f"Language: {result.input_language}")
print(f"Intent:   {result.intent}")
print(f"Key text: {result.key_text}")
print(f"\n📝 Analysis:\n{result.analysis}")
print(f"\n💬 Response:\n{result.response}")

Language: hiligaynon
Intent:   define
Key text: 'maayo'

📝 Analysis:
The input contains the Hiligaynon word "maayo", which means "good" in English. The phrase structure and vocabulary used are typical of Hiligaynon. The use of "ano ang buot silingon" is a common way to ask for the meaning of a word in Hiligaynon.

💬 Response:
The Hiligaynon word "maayo" can be translated to English as "good". However, its meaning can vary depending on the context in which it is used. For example, it can describe something as "beautiful" or "handsome", as seen in the definitions of "maayó-áyo" and "maányag". It can also be used to express that something is "excellent" or of high quality. Additionally, "maayo" can be used to describe someone's character, meaning they are "virtuous", "honest", or "kind". In some cases, it can also be used to express a sense of wellness or health, as in "maáyo siá sing láwas", which means "he is physically fit" or "he is healthy". It's worth noting that Hiligaynon has vowe

## 4. Test: Vowel Variant Awareness (o↔u, i↔e)

In [5]:
# Test vowel variant awareness — 'buot' vs 'boot' should give similar results
import time

vowel_tests = [
    "What does 'buot' mean?",
    "What does 'boot' mean in Hiligaynon?",
    "Translate 'diin' to English",
    "What is 'deen' in English?",
]

for msg in vowel_tests:
    print(f"\n{'='*60}")
    print(f"📨 {msg}")
    result = chatbot(user_message=msg)
    print(f"   Key text: {result.key_text}")
    print(f"   💬 {result.response[:200]}...")
    time.sleep(2)  # Rate limiting for Groq free tier


📨 What does 'buot' mean?
   Key text: buot
   💬 The word "buot" is likely related to the Hiligaynon term for footwear, possibly being an alternative spelling or pronunciation of "bótas" or "sapín". In Hiligaynon, the vowels "o" and "u" are often us...

📨 What does 'boot' mean in Hiligaynon?
   Key text: boot
   💬 The Hiligaynon translation for the English word "boot" is "bótas". This term directly refers to a type of footwear that includes both boots and shoes, reflecting the Spanish influence on the Hiligayno...

📨 Translate 'diin' to English
   Key text: diin
   💬 The Hiligaynon word "diin" translates to "where" in English. It is used to ask about the location of something or someone, similar to how "where" is used in English. For example, "Diin ka makadto?" me...

📨 What is 'deen' in English?
   Key text: deen
   💬 It seems like 'deen' might be a variation of the Hiligaynon word 'diin', which means 'where' or 'at what place'. In Hiligaynon, vowels like 'e' and 'i' are often used in

## 5. Test: DSPy Evaluator Compatibility

The chatbot also accepts `hiligaynon`/`english` kwargs directly (for DSPy evaluation).

In [6]:
# Simulate DSPy evaluator calling with field arguments directly
result = chatbot(hiligaynon="balay", english="house")

print(f"Language: {result.input_language}")
print(f"Intent:   {result.intent}")
print(f"Key text: {result.key_text}")
print(f"\n💬 Response:\n{result.response}")

Language: mixed
Intent:   define
Key text: balay

💬 Response:
The Hiligaynon word "balay" translates to "house" or "home" in English. It can also refer to other types of dwellings or buildings, such as a nest or a shell. It's interesting to note that the word "balay" has a range of related meanings and uses in Hiligaynon, including the diminutive form "baláy-bálay" which refers to a small dwelling like a hut or shack. In general, "balay" is a fundamental word in Hiligaynon that is used to describe a person's place of residence or a structure that serves as a home. For example, you might ask "Diín ang baláy mo?" which means "Where is your home?" or say "Ang baláy sang ánay" which refers to "The nest of termites". The word "balay" is a key part of Hiligaynon vocabulary and is used in a variety of contexts to convey the idea of a dwelling or home.


## 6. Run Baseline Evaluation

In [7]:
from src.metrics import translation_relevance_metric, load_examples

# Load dev examples
dev_examples = load_examples(ROOT / "data" / "examples" / "devset.json")
print(f"Loaded {len(dev_examples)} dev examples")

# Quick test with one example
ex = dev_examples[0]
print(f"\nTest example: hiligaynon='{ex.hiligaynon}', english='{ex.english}'")

pred = chatbot(hiligaynon=ex.hiligaynon, english=ex.english)
score = translation_relevance_metric(ex, pred)
print(f"Score: {float(score):.2f}")
print(f"Response: {pred.response[:200]}")

Loaded 30 dev examples

Test example: hiligaynon='adlaw', english='day; sun'
Score: 0.70
Response: The Hiligaynon term 'adlaw' can be translated to English as 'day' or 'sun'. Interestingly, this word has multiple uses in the language, such as referring to daylight, a 24-hour period, or even a speci
